# 02 - Data Preprocessing

Cleans the raw load log and segments it into fixed 10-second intervals, matching
`algorithms/load_monitoring_algorithm.md` and thesis Figure 3.3.3. Uses
`src/preprocessing.py` directly (not reimplemented here).

In [ ]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from preprocessing import clean_load_log, segment_into_intervals

df = pd.read_csv("../data/synthetic_load_log.csv")
df = clean_load_log(df)
print("After cleaning:", len(df), "rows")
df.head()


In [ ]:
intervals = segment_into_intervals(df[["timestamp", "applied_load_kg"]], interval_seconds=10)
# Carry the location forward using the nearest raw reading for each interval
# (merge_asof handles duplicate timestamps in the raw log; a plain reindex does not).
loc_lookup = df[["timestamp", "unit_location"]].sort_values("timestamp").drop_duplicates("timestamp")
intervals = pd.merge_asof(
    intervals.sort_values("timestamp"), loc_lookup, on="timestamp", direction="nearest"
)
intervals["day_of_week"] = pd.to_datetime(intervals["timestamp"]).dt.day_name()
intervals = intervals.dropna(subset=["max_load_kg"])
print(len(intervals), "intervals")
intervals.head()


In [ ]:
intervals.to_csv("../data/intervals.csv", index=False)
print("Saved ../data/intervals.csv")
